# Neural Network in PyTorch
A feed-forward network trained on the digits dataset (8x8 handwritten digit images, flattened to 64 features), covering tensors, a custom Dataset, DataLoader, and a loss function.

In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Load and prepare the data
Split into train/test, scale the features, then wrap everything in a PyTorch Dataset and DataLoader.

In [10]:
class DigitsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [11]:
data = load_digits()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_ds = DigitsDataset(X_train, y_train)
test_ds = DigitsDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print(len(train_ds), "training samples,", len(test_ds), "test samples")

1437 training samples, 360 test samples


## Define the network
Two hidden layers with ReLU activations, ending in a 10-class output for digits 0-9.

In [12]:
class FeedForwardNet(nn.Module):
    def __init__(self, input_size=64, hidden_size=32, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = FeedForwardNet()
print(model)

FeedForwardNet(
  (net): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=10, bias=True)
  )
)


## Train
Using Adam as the optimizer and cross entropy as the loss function, since this is a multi-class classification problem.

In [13]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

epochs = 10
for epoch in range(epochs):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = loss_fn(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1}/{epochs}, loss: {total_loss/len(train_loader):.4f}")

epoch 1/10, loss: 2.1671
epoch 2/10, loss: 1.5347
epoch 3/10, loss: 0.7519
epoch 4/10, loss: 0.3997
epoch 5/10, loss: 0.2600
epoch 6/10, loss: 0.1869
epoch 7/10, loss: 0.1434
epoch 8/10, loss: 0.1142
epoch 9/10, loss: 0.0924
epoch 10/10, loss: 0.0775


## Evaluate on the test set

In [14]:
correct = 0
total = 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        predicted = torch.argmax(outputs, dim=1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

accuracy = correct / total
print(f"test accuracy: {accuracy:.4f}")

test accuracy: 0.9639


In [15]:
torch.save(model.state_dict(), "feedforward_model.pt")
print("model saved")

model saved
